In [1]:
import logging
from itertools import combinations

import pandas as pd
import numpy as np
from scipy.stats import spearmanr


from etl_utils import(
    create_adocell_prefix,
    load_results,
    AIDOCELL_CLASSES_LIST,
    MODELS,
    ONTOLOGIES,
    RESULTS_DEFS
) 

from analysis_utils import (
    load_foundation_summaries,
    get_common_identifiers,
    get_aligned_embeddings,
    select_device,
    compute_cosine_distances_torch,
    compute_spearman_correlation_torch,
)


# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s:%(name)s:%(message)s')
logger = logging.getLogger(__name__)

aido_cell_prefixes = {create_adocell_prefix(x) for x in AIDOCELL_CLASSES_LIST}
model_prefixes = {MODELS.SCPRINT, MODELS.SCGPT} | aido_cell_prefixes

OUTPUT_DIR = "output"


In [2]:
# load each model's weights, gene annotations, and model metadata
models_dict = load_foundation_summaries(model_prefixes, OUTPUT_DIR)

# Get common identifiers across all models
common_identifiers = get_common_identifiers(models_dict)

# pull out and align embeddings across models
aligned_embeddings = get_aligned_embeddings(models_dict, common_identifiers)

INFO:analysis_utils:Loading results for the scPRINT foundation model
INFO:etl_utils:Loading weights from output/scPRINT_weights.npz and metadata from output/scPRINT_metadata.json
INFO:etl_utils:Loading weights from output/scPRINT_weights.npz
INFO:etl_utils:Loading metadata from output/scPRINT_metadata.json
INFO:etl_utils:Successfully loaded and validated all results
INFO:analysis_utils:Loading results for the AIDOCell_aido_cell_10m foundation model
INFO:etl_utils:Loading weights from output/AIDOCell_aido_cell_10m_weights.npz and metadata from output/AIDOCell_aido_cell_10m_metadata.json
INFO:etl_utils:Loading weights from output/AIDOCell_aido_cell_10m_weights.npz
INFO:etl_utils:Loading metadata from output/AIDOCell_aido_cell_10m_metadata.json
INFO:etl_utils:Successfully loaded and validated all results
INFO:analysis_utils:Loading results for the scGPT foundation model
INFO:etl_utils:Loading weights from output/scGPT_weights.npz and metadata from output/scGPT_metadata.json
INFO:etl_utils

In [3]:
# Check if MPS/GPU is available
device = select_device(mps_valid = True) # MPS is slower than CPU for this task
logger.info(f"Using device: {device}")

# Convert embeddings to PyTorch tensors and compute distances
distances = {}
for model_name, embedding in aligned_embeddings.items():
    logger.info(f"Computing distances for {model_name}...")
    distances[model_name] = compute_cosine_distances_torch(embedding, device)

# Compare distance matrices pairwise - all unique pairs from model_prefixes
# Use upper triangle only (exclude diagonal and avoid redundancy)
mask = np.triu_indices(len(common_identifiers), k=1)  # k=1 excludes diagonal

comparisons = {}
for model1, model2 in combinations(model_prefixes, 2):
    logger.info(f"Comparing {model1} vs {model2}...")
    
    dist1_flat = distances[model1][mask]
    dist2_flat = distances[model2][mask]
    
    # Spearman correlation using PyTorch 
    rho = compute_spearman_correlation_torch(dist1_flat, dist2_flat, device)
    comparisons[f"{model1}_vs_{model2}"] = rho
    logger.info(f"  {model1} vs {model2}: Spearman rho = {rho:.4f}")

print(f"\nFound {len(comparisons)} unique pairwise comparisons:")
for comparison, rho in comparisons.items():
    print(f"  {comparison}: {rho:.4f}")

INFO:__main__:Using device: mps
INFO:__main__:Computing distances for scPRINT...
INFO:__main__:Computing distances for AIDOCell_aido_cell_10m...
INFO:__main__:Computing distances for scGPT...
INFO:__main__:Computing distances for AIDOCell_aido_cell_3m...
INFO:__main__:Computing distances for AIDOCell_aido_cell_100m...
INFO:__main__:Comparing scPRINT vs AIDOCell_aido_cell_10m...
INFO:__main__:  scPRINT vs AIDOCell_aido_cell_10m: Spearman rho = 0.0192
INFO:__main__:Comparing scPRINT vs scGPT...
INFO:__main__:  scPRINT vs scGPT: Spearman rho = 0.2768
INFO:__main__:Comparing scPRINT vs AIDOCell_aido_cell_3m...
INFO:__main__:  scPRINT vs AIDOCell_aido_cell_3m: Spearman rho = 0.0253
INFO:__main__:Comparing scPRINT vs AIDOCell_aido_cell_100m...
INFO:__main__:  scPRINT vs AIDOCell_aido_cell_100m: Spearman rho = 0.0047
INFO:__main__:Comparing AIDOCell_aido_cell_10m vs scGPT...
INFO:__main__:  AIDOCell_aido_cell_10m vs scGPT: Spearman rho = 0.0263
INFO:__main__:Comparing AIDOCell_aido_cell_10m v


Found 10 unique pairwise comparisons:
  scPRINT_vs_AIDOCell_aido_cell_10m: 0.0192
  scPRINT_vs_scGPT: 0.2768
  scPRINT_vs_AIDOCell_aido_cell_3m: 0.0253
  scPRINT_vs_AIDOCell_aido_cell_100m: 0.0047
  AIDOCell_aido_cell_10m_vs_scGPT: 0.0263
  AIDOCell_aido_cell_10m_vs_AIDOCell_aido_cell_3m: 0.0025
  AIDOCell_aido_cell_10m_vs_AIDOCell_aido_cell_100m: 0.0005
  scGPT_vs_AIDOCell_aido_cell_3m: 0.0376
  scGPT_vs_AIDOCell_aido_cell_100m: 0.0052
  AIDOCell_aido_cell_3m_vs_AIDOCell_aido_cell_100m: 0.0004
